# 02 — SLA & stockout baseline

Lag-7 seasonal naive on delivered orders + zone SLA coaching list.

Not a production forecaster — enough to show zone / SKU pressure.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..')
MARTS = ROOT / 'data' / 'marts'
OUT = Path('outputs')
REPORTS = ROOT / 'reports'
OUT.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(MARTS / 'fact_orders.csv', parse_dates=['order_date'])
daily = pd.read_csv(MARTS / 'mart_daily.csv', parse_dates=['order_date'])
zone = pd.read_csv(MARTS / 'mart_zone.csv')
so = pd.read_csv(MARTS / 'mart_top_stockout_skus.csv')
delivered = orders[orders['is_delivered'] == 1].copy()
print('delivered', len(delivered), 'zones', len(zone))


In [ ]:
zone_rank = zone.sort_values('sla_hit_pct')
worst = zone_rank.head(5)
best = zone_rank.tail(5)

d = daily.sort_values('order_date').copy()
d['yhat'] = d['delivered_orders'].shift(7)
hold = d.dropna(subset=['yhat']).copy()
hold['ape'] = (hold['delivered_orders'] - hold['yhat']).abs() / hold['delivered_orders'].clip(lower=1)
mape = float(hold['ape'].mean())
top10_share = float(so.head(10)['stockout_events'].sum() / so['stockout_events'].sum()) if len(so) else 0.0
slot_sla = delivered.groupby('slot')['sla_hit'].mean().to_dict() if 'slot' in delivered.columns and 'sla_hit' in delivered.columns else {}

metrics = {
    'demand_baseline': 'seasonal_naive_lag7',
    'demand_mape': round(mape, 4),
    'holdout_days': int(len(hold)),
    'worst_zones': worst[['metro', 'zone', 'sla_hit_pct', 'stockout_rate', 'o2d_p50']].to_dict('records') if set(['metro','zone','sla_hit_pct']).issubset(worst.columns) else worst.head(5).to_dict('records'),
    'best_zones': best.tail(5).to_dict('records'),
    'top10_sku_stockout_share': round(top10_share, 4),
    'slot_sla_hit_pct': {k: round(float(v), 4) for k, v in slot_sla.items()},
    'notes': ['Lag-7 naive is a baseline only — no weather / promo features.'],
}
(REPORTS / 'sla_stockout_metrics.json').write_text(json.dumps(metrics, indent=2, default=str))
print('MAPE', metrics['demand_mape'])
worst


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(d['order_date'], d['delivered_orders'], label='delivered')
ax.plot(d['order_date'], d['yhat'], linestyle='--', label='lag-7')
ax.set_title('Daily delivered vs lag-7 naive')
ax.legend()
ax.tick_params(axis='x', labelrotation=45, labelsize=7)

ax2 = axes[1]
zplot = zone_rank.head(8)
labels = (zplot['metro'].astype(str) + ' / ' + zplot['zone'].astype(str)) if 'metro' in zplot.columns else zplot.index.astype(str)
ax2.barh(labels, zplot['sla_hit_pct'], color='#F87171')
ax2.set_title('Lowest SLA zones')
fig.tight_layout()
fig.savefig(OUT / 'sla_stockout_baseline.png', dpi=140)
plt.show()
